In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, lit

In [0]:
# Creating File Registry table --- This is only for Raw Layer
sql = '''
CREATE or replace TABLE zapflow_control.control.file_registry (
    file_name      STRING,
    domain         STRING,
    file_status    STRING,
    file_size      LONG,
    file_path      STRING,
    batch_id       STRING,
    registered_at  TIMESTAMP,
    processed_at   TIMESTAMP,
    row_count      LONG
)
'''
spark.sql(sql)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

file_registry_schema = StructType([
    StructField('file_name',      StringType(),    True),
    StructField('domain',         StringType(),    True),
    StructField('file_status',    StringType(),    True),
    StructField('file_size',      LongType(),      True),
    StructField('file_path',      StringType(),    True),
    StructField('batch_id',       StringType(),    True),
    StructField('registered_at',  TimestampType(), True),
    StructField('processed_at',   TimestampType(), True),
    StructField('row_count',      LongType(),      True)
])

In [0]:
%skip
#Get Customers
ls = dbutils.fs.ls('/Volumes/zapflow_raw/raw_data/raw_files/customers/')
files = [file for file in ls if file.name.endswith('.csv')]
# print(files)
details = []
for file in files:
    details.append([file.name, 'customers', 'pending', file.modificationTime,None ,  file.size, file.path])
df = spark.createDataFrame([], file_registry_schema)
for detail in details:
    df = df.union(spark.createDataFrame([detail], file_registry_schema))
df.write.mode('append').saveAsTable('zapflow_raw.control.file_registry')

In [0]:
from datetime import datetime
def register_pending_files(domain, source_path):
    #get all the csv files from given path
    all_files = dbutils.fs.ls(source_path)
    files = [f for f in all_files if f.name.endswith('.csv')]

    #check whether the files are present in the register
    files_registered = spark.sql(f"select file_path from zapflow_raw.control.file_registry where domain = '{domain}'")

    #get the unregistered files
    new_files = [f for f in files if f.path not in files_registered.select('file_path').collect()]

    #register new files

    if new_files:
        now = datetime.now()
        rows = [
            (
                f.name,
                domain,
                'PENDING',
                f.size,
                f.path,
                None,        # batch_id — not assigned yet
                now,         # registered_at
                None,        # processed_at — not done yet
                None         # row_count — not written yet
            )
            for f in new_files
        ]
        df = spark.createDataFrame(rows, file_registry_schema)
        df.write.mode('append').saveAsTable('zapflow_raw.control.file_registry')






In [0]:
# 1. Get files that haven't been processed yet
def get_pending_files(domain, source_path):
    sql = f"select file_path from zapflow_control.control.file_registry where domain = '{domain}' and file_status = 'PENDING'"
    pending_files = spark.sql(sql)
    return pending_files
    # print(list)

# 2. Mark a file as in progress
def mark_in_progress(file_path, batch_id):
    sql = f"update zapflow_control.control.file_registry set file_status = 'IN_PROGRESS', batch_id = '{batch_id}' where file_path = '{file_path}'"
    spark.sql(sql)
    return


# 3. Mark a file as completed
def mark_completed(file_path, row_count):
    sql = f"update zapflow_control.control.file_registry set file_status = 'COMPLETED', processed_at = current_timestamp(), row_count = {row_count} where file_path = '{file_path}'"
    spark.sql(sql)
    return
